In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sklearn.datasets import load_iris
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC



# Arboles de Decisión y métodos de Aprendizaje Conjunto


En este Notebook vamos a ir implementando los contenidos y ejemplos de la presentación.

#### Cálculo de la Impureza de Gini y de la Entropía.

In [ ]:
def Gini(y):
    n_classes = len(np.unique(y)) 
    n_samples = len(y)

    value = 1.0
    for class_k in np.unique(y):
        num_class_k = n_samples - np.count_nonzero(y-class_k)
        ratio = num_class_k/n_samples
        value -= ratio**2
    return value

def Entropia(X,y):
    n_classes = len(np.unique(y)) 
    n_samples = len(y)

    value = 0.0
    for class_k in np.unique(y):
        num_class_k = n_samples - np.count_nonzero(y-class_k)
        ratio = num_class_k/n_samples
        print(class_k,num_class_k, ratio)
        if ratio>0.00001:
            value += ratio*np.log2(ratio)
    return -value

Z = np.linspace(1e-10, 1-1e-10, 100)
Gini_z = 1-Z**2-(1-Z)**2
Entropia_z = 0.5*(-Z*np.log2(Z)-(1-Z)*np.log2(1-Z))

plt.figure(figsize=(6, 4))
plt.plot(Z, Gini_z,  label ="Impureza de Gini ")
plt.plot(Z, Entropia_z , label ="Medida de Entropia ")
plt.xlim(0,1)
plt.xlabel("Fracción de la clase", fontsize=14)
plt.legend(loc="lower center", fontsize=10)

De forma manual implementamos una función que dado un conjunto de datos y sus etiquetas correspondientes calcula la variable de entrada y el umbral que generan la partición optima del nodo. Aquella que maximiza la ganancia de información (Minimiza la Impureza de Gini en la división)

In [ ]:
def find_best_split(X,y):
    n_samples, n_features = X.shape
    IG = 0.0
    
    # Entropia inicial
    Gini_padre = Gini(y)
    
    for i_feature in range(n_features):
        feature_min = np.min(X[:, i_feature])
        feature_max = np.max(X[:, i_feature])
        list_threshold = np.linspace(feature_min, feature_max, 20, endpoint = False)
        for i_threshold in list_threshold:
            
            array_index = X[:,i_feature]<=i_threshold
            X_left = X[array_index]
            n_left = sum(array_index)
            y_left = y[array_index]
            X_right = X[~array_index]
            n_right = n_samples-n_left
            y_right = y[~array_index]
            
            Gini_hijos = (n_left/n_samples)*Gini(y_left) + (n_right/n_samples)*Gini(y_right)
            
            if Gini_padre - Gini_hijos > IG:
                IG = Gini_padre - Gini_hijos
                best_feature = i_feature
                best_threshold = i_threshold
    return i_feature, i_threshold

Voy a utilizar el conjunto de datos *Iris Dataset* para ver la implementación de los arboles de decisión de forma *casera* y con las funciones predefinidas dentro de `Sklearn`.

Empecemos cargando los datos como ya hemos hecho otras veces.

In [ ]:
iris = load_iris()
X = iris.data[:, 2:] # petal length and width
y = iris.target

Ahora aplico al conjunto de datos *Iris* la función que determina el par *variable/valor* optimo para dividir el conjunto original

In [ ]:
print(' Indice de Gini en el conjunto original:',Gini(y))
best_feature, best_threshold = find_best_split(X,y)
print(f' Variable y valor sobre la que haremos la división: {iris.feature_names[1+best_feature]}={best_threshold}')

plt.figure(figsize=(8, 4))
plt.plot(X[y==0, 0], X[y==0, 1], "bo", markersize=7, fillstyle= 'none', label ="Iris Setosa ")
plt.plot(X[y==1, 0], X[y==1, 1], "g^", markersize=7, fillstyle= 'none', label ="Iris Versicolor ")
plt.plot(X[y==2, 0], X[y==2, 1], "rs", markersize=6, fillstyle= 'none', label ="Iris Virginica")
plt.plot([best_threshold, best_threshold], [0, 3], "k-", linewidth=2)
plt.ylim(0,3)
plt.xlabel("Longitud del pétalo", fontsize=14)
plt.ylabel("Anchura del pétalo", fontsize=14)
plt.legend(loc="upper left", fontsize=10)

## Arboles de decisión con *SkLearn*

La clase `tree.DecisionTreeClassifier` de `scikit-learn` tiene varios parámetros importantes que afectan el entrenamiento y la estructura del árbol de decisión.

* `criterion` (Criterio de impureza). Define cómo se mide la impureza del nodo para realizar la división.
    * *gini* (por defecto) → Usa el índice de Gini para medir la impureza.
    * *entropy* → Usa la entropía (logaritmo de la probabilidad) para medir la impureza.

* `max_depth` (Profundidad máxima del árbol). Controla cuántos niveles puede tener el árbol.
    *  *None* (por defecto), crece hasta que todas las hojas sean puras o haya pocas muestras.
    * Valores más bajos → Evitan el sobreajuste.
    * Valores más altos → Permiten capturar más patrones, pero pueden sobreajustar.

* `min_samples_split` (Muestras mínimas para dividir un nodo). Indica cuántas muestras como mínimo debe tener un nodo para dividirse.
    * Valor por defecto: 2.
    * Valores más altos → Árbol más simple y menos sobreajuste.

* `min_samples_leaf` (Muestras mínimas en una hoja). Define el mínimo de muestras que debe haber en cada nodo hoja.
    * Evita que el árbol cree hojas con muy pocas muestras (reduce sobreajuste).

* `max_features` (Máximo de características usadas por nodo).Controla cuántas variables considera el árbol en cada división.
    * *None* → Usa todas las características (por defecto).
    * *sqrt* → Usa la raíz cuadrada del número total de características (bueno para Random Forest).
    * *log2* → Usa log2(n_features) características.

* `class_weight` (Peso de las clases). Se usa para balancear clases desbalanceadas.
    * *None* (por defecto) → Todas las clases tienen el mismo peso.
    * *balanced* → Ajusta los pesos de forma automática.
    * También se puede definir manualmente.


El proceso de entrenamiento y predicción en un árbol de decisión con scikit-learn sigue dos pasos:

* `fit(X, y)`: Entrenamiento del árbol. Toma los datos de entrada (*X*) y las etiquetas (*y*) y construye el árbol de decisión.
* `predict(X)`: Predicción con nuevos datos. Una vez entrenado el árbol, podemos usar predict(X) para hacer predicciones sobre datos nuevos.
* `predict_proba(X)`: Probabilidad de clases. Devuelve la probabilidad de cada clase en lugar de la predicción de cada clase.

El método `plot_tree` de `sklearn.tree` nos permite visualizar un árbol de decisión entrenado de forma gráfica. Es una herramienta útil para entender cómo el modelo toma decisiones y ver las reglas de partición.

#### Cómo leer el gráfico
Cada nodo en el árbol muestra información clave:

* Feature usada para la división (sepal width <= 2.45)
* Valor de impureza (ej. Gini o Entropía)
* Cantidad de muestras en el nodo
* Distribución de clases en ese nodo

In [ ]:
X_train, X_test, y_train, y_test =  train_test_split(X,y,test_size=0.3, random_state=231)

clf = DecisionTreeClassifier(max_depth=2, random_state=231)
clf = clf.fit(X_train, y_train)

# Graficar el árbol
plt.figure(figsize=(12, 8))  # Ajustar tamaño
plot_tree(clf, filled=True, feature_names=iris.feature_names, class_names=iris.target_names)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.plot(X_train[y_train==0, 0], X_train[y_train==0, 1], "bo", markersize=7, fillstyle= 'none', label ="Iris Setosa ")
plt.plot(X_train[y_train==1, 0], X_train[y_train==1, 1], "g^", markersize=7, fillstyle= 'none', label ="Iris Versicolor ")
plt.plot(X_train[y_train==2, 0], X_train[y_train==2, 1], "rs", markersize=6, fillstyle= 'none', label ="Iris Virginica")
plt.plot([0, 7], [0.75, 0.75], "k-", linewidth=2)
plt.plot([4.95, 4.95], [0.75, 3], "k--", linewidth=2)
"""
plt.plot([4.95, 4.95], [0, 1.75], "k--", linewidth=2)
plt.plot([2.45, 4.95], [1.45, 1.45], "k--", linewidth=1)
plt.plot([4.95, 4.95], [1.75, 3.], "k--", linewidth=2)
"""
plt.xlim(0,7)
plt.ylim(0,3)
plt.xlabel("Longitud del pétalo", fontsize=14)
plt.ylabel("Anchura del pétalo", fontsize=14)
plt.legend(loc="upper left", fontsize=10)

### Arboles de Regresión con SKLearn

In [ ]:
np.random.seed(42)
m = 300
X = np.random.rand(m, 1)
y = np.sin(2*np.pi*X)
y = y + np.random.randn(m, 1) / 10

plt.plot(X, y, "b.")
plt.xlim(-0.01,1.01)
plt.ylim(-1.4,1.4)
plt.xlabel("X", fontsize=10)
plt.ylabel("y", fontsize=10, rotation= 0)
#plt.legend(loc="upper right", fontsize=10)

 La clase `DecisionTreeRegressor` de `scikit-learn` se usa para regresión, es decir, predecir valores continuos en lugar de categorías. Es la versión para regresión de `DecisionTreeClassifier`, y funciona dividiendo los datos en regiones y asignando un valor promedio a cada una.

 * `criterion` 	Función de error para dividir nodos 
    * *squared_error* (por defecto).
    * *absolute_error* 

In [ ]:
tree_reg = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_reg.fit(X, y)

# Graficar el árbol
plt.figure(figsize=(12, 8))  # Ajustar tamaño
plot_tree(tree_reg, filled=True, rounded=True)
plt.show()


In [ ]:
np.random.seed(42)
m = 300
X = np.random.rand(m, 1)
y = np.sin(2*np.pi*X)
y = y + np.random.randn(m, 1) / 10

tree_reg_2 = DecisionTreeRegressor(max_depth=2, random_state=142)
tree_reg_4 = DecisionTreeRegressor(max_depth=4, random_state=142)
tree_reg_8 = DecisionTreeRegressor(max_depth=8, random_state=142)
tree_reg_no_limit = DecisionTreeRegressor(random_state=142)
tree_reg_2.fit(X, y)
tree_reg_4.fit(X, y)
tree_reg_8.fit(X, y)
tree_reg_no_limit.fit(X, y)

X_test = np.linspace(0,1,400).reshape(400,1)
y_2 = tree_reg_2.predict(X_test)
y_4 = tree_reg_4.predict(X_test)
y_8 = tree_reg_8.predict(X_test)
y_no_limit = tree_reg_no_limit.predict(X_test)

#plot_tree(tree_reg_2, filled=True, rounded=True)

# Plot the results


# Crear figura y ejes en formato 2x2
fig, axes = plt.subplots(2, 2, figsize=(12, 10))  # 2 filas, 2 columnas


axes[0,0].plot(X, y, "b.", label="Datos de entrenamiento")  # Datos originales
axes[0,0].plot(X_test, y_2, color="red", label="max_depth=2", linewidth=2)  # Predicción
axes[0,0].set_xlabel("X")
axes[0,0].set_ylabel("y", rotation=0)
axes[0,0].set_title('max_depth = 2')
axes[0,0].legend()

axes[0,1].plot(X, y, "b.", label="Datos de entrenamiento")  # Datos originales
axes[0,1].plot(X_test, y_4, color="red", label="max_depth=4", linewidth=2)  # Predicción
axes[0,1].set_xlabel("X")
axes[0,1].set_ylabel("y", rotation=0)
axes[0,1].set_title('max_depth = 4')
axes[0,1].legend()

axes[1,0].plot(X, y, "b.", label="Datos de entrenamiento")  # Datos originales
axes[1,0].plot(X_test, y_8, color="red", label="max_depth=8", linewidth=2)  # Predicción
axes[1,0].set_xlabel("X")
axes[1,0].set_ylabel("y", rotation=0)
axes[1,0].set_title('max_depth = 8')
axes[1,0].legend()

axes[1,1].plot(X, y, "b.", label="Datos de entrenamiento")  # Datos originales
axes[1,1].plot(X_test, y_no_limit, color="red", label="sin límite de profundidad", linewidth=2)  # Predicción
axes[1,1].set_xlabel("X")
axes[1,1].set_ylabel("y", rotation=0)
axes[1,1].set_title('Sin restricciones')
axes[1,1].legend()

plt.tight_layout()
plt.show()

# Métodos de Aprendizaje Conjunto.

Aquí vamos a ver dos tipos:

* Bosques aleatorios (*Random Forest*) como un conjunto de árboles de decisión.
* Métodos de *ensemble* con modelos diferentes.

## Random Forest con Scikit-Learn 

La clase `RandomForestClassifier` de `scikit-learn` es un modelo de aprendizaje conjunto basado en árboles de decisión.

Parametros:

* `n_estimators`: Número de árboles en el bosque.	100-500 (por defecto: 100)
* `criterion`:	Función para medir la calidad de la división.	
        * *gini* 
        *  *entropy* 
* `max_depth`	Profundidad máxima de cada árbol (evita sobreajuste).	5-20 (por defecto: None)
* `min_samples_split`	Muestras mínimas necesarias para dividir un nodo.	2-10
* `min_samples_leaf`	Muestras mínimas en cada nodo hoja.	1-5
* `max_features`	Número de características aleatorias por árbol.	"sqrt", "log2", o número específico
* `bootstrap`	Si True, usa muestreo con reemplazo.	True (por defecto)

In [ ]:
iris = load_iris()
X = iris.data  # petal length and width
y = iris.target

X_train, X_test, y_train, y_test =  train_test_split(X,y,test_size=0.6, random_state=231)

rf=RandomForestClassifier(n_estimators=10, max_depth=2, random_state=42)
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
print(accuracy_score(y_test, y_pred))

### Feature Importance

Una ventaja de usar `Sklearn` es que podemos obtener la importancia de cada característica de entrada en la clasificación directamente usando 
```python
.feature_importances_
```

In [ ]:
# Obtener la importancia de las características
importances = rf.feature_importances_

# Graficar la importancia
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title("Importancia de las características")
plt.bar(range(X.shape[1]), importances[indices], align="center")
plt.xticks(range(X.shape[1]), np.array(iris.feature_names)[indices], rotation=90)
plt.xlim([-1, X.shape[1]])
plt.show()

for name, score in zip(iris["feature_names"], rf.feature_importances_):
    print(name, score)

### Pasting vs Bagging.

Por defecto el modelo emplea *Bagging: Bootstrap* sobre el conjunto de datos de entrenamiento para alimentar cada arbol dentro del bosque. Veamos el efecto de forzar *Pasting*

* Cuándo considerar usar bootstrap (generalmente recomendado):

  * Conjuntos de datos más pequeños: El bootstrapping ayuda a reducir la varianza y los riesgos de sobreajuste de manera más significativa cuando se entrena con conjuntos de datos pequeños.

  * Datos de alta dimensión: Con muchas características, el bootstrapping puede ayudar a mejorar la generalización del modelo al inyectar aleatoriedad.

  * Datos altamente variables: Si tus datos exhiben una variación significativa, el bootstrapping puede promediar los efectos de los valores atípicos y mejorar la robustez del modelo.

* Cuándo podrías considerar no usar bootstrap:

  * Conjuntos de datos grandes: Para conjuntos de datos muy grandes, el costo computacional del bootstrapping podría superar los beneficios. En tales casos, utilizar un conjunto de validación para la estimación de errores podría ser una mejor opción.

  * Datos de baja dimensión y estables: Si tienes un conjunto de datos limpio con baja dimensionalidad y variación mínima, el bootstrapping podría no ser necesario. 

In [ ]:
# Forzar pasting (sin reemplazo)
rf_pasting = RandomForestClassifier(bootstrap=False, n_estimators=10, max_depth=2, random_state=42)

# Entrenar el modelo
rf_pasting.fit(X_train, y_train)
y_pred = rf_pasting.predict(X_test)
print(f'Accuracy usando pasting (sin reemplazo) :{accuracy_score(y_test, y_pred)}')
y_pred = rf.predict(X_test)
print(f'Accuracy usando bagging (con reemplazo) :{accuracy_score(y_test, y_pred)}')

### *Hard vs Soft Voting*

Por defecto los metodos de **Aprendizaje Conjunto** en `Sklearn` emplean el voto por mayoría. 
Utilizar voto ponderado en *Random Forest* pasa por utilizar el método `.predict_proba()` para obtener la probabilidad de pertenencia a cada clase. 
Veamos como hacerlo en el ejemplo de *Iris Dataset* usano solo las variables menos explicativas.

In [ ]:
# Datos
iris = load_iris()
X = iris.data[:, :2]
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=142, stratify=y
)

# Entrenar RF
rf = RandomForestClassifier(n_estimators=15, max_depth=2, random_state=142)
rf.fit(X_train, y_train)

# --- HARD voting entre árboles ---
# Predicción de cada árbol (n_estimators, n_samples)
tree_preds = np.vstack([est.predict(X_test) for est in rf.estimators_])

# Mayoría por columna
y_pred_hard = np.apply_along_axis(
    lambda col: np.bincount(col.astype(int)).argmax(),
    axis=0,
    arr=tree_preds
)

# --- SOFT voting entre árboles ---
# Probabilidades de cada árbol (n_estimators, n_samples, n_classes)
tree_probas = np.stack([est.predict_proba(X_test) for est in rf.estimators_])
avg_proba = tree_probas.mean(axis=0)
y_pred_soft = avg_proba.argmax(axis=1)

# Evaluación
accuracy_hard = accuracy_score(y_test, y_pred_hard)
accuracy_soft = accuracy_score(y_test, y_pred_soft)

print(f"Precisión (Hard voting entre árboles): {accuracy_hard:.4f}")
print(f"Precisión (Soft voting entre árboles): {accuracy_soft:.4f}")
print("¿Predicciones iguales?", np.array_equal(y_pred_hard, y_pred_soft))



## Métodos de *Ensemble* con Scikit-Learn 

La clase `VotingClassifier` de scikit-learn es un modelo de aprendizaje conjunto que combina múltiples clasificadores para mejorar la precisión y la estabilidad de las predicciones. 

Combina varios modelos de clasificación en un solo modelo. Cada modelo hace una predicción sobre una muestra nueva. El modelo final decide por votación:

* Votación mayoritaria (hard) → La clase con más votos gana.
* Votación ponderada (soft) → Promedio de probabilidades de los modelos

Los parámetros más importantes son:

* `estimators`:	Lista de modelos base con un alias `[('lr', model1), ('rf', model2)]`.
* `voting`:
    * 	*hard* para votación mayoritaria  
    *  *soft* para promedio de probabilidades.
* `weights`:	ista con pesos para cada modelo en votación *soft* (ejemplo: [2, 1, 3]).

Veamos un ejemplo en una clasificación binaria.

In [ ]:
X, y = make_moons(n_samples=500, noise=0.20, random_state=342)

plt.plot(X[:, 0][y==0], X[:, 1][y==0], "gD", ms = 4)
plt.plot(X[:, 0][y==1], X[:, 1][y==1], "bo", ms = 4)
plt.xlabel(r"$x_1$", fontsize=10)
plt.ylabel(r"$x_2$", fontsize=10, rotation=0)
plt.title("Semicirculos imbricados", fontsize=14)

plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=342)

In [ ]:
log_clf = LogisticRegression(solver="lbfgs", random_state=1242)
rnd_clf = RandomForestClassifier(n_estimators=10, random_state=1242)
svm_clf = SVC(gamma="scale", probability=True, random_state=1242)
tree1_clf = DecisionTreeClassifier(max_depth=4, random_state=1242)
tree2_clf = DecisionTreeClassifier(max_depth=8, random_state=1242)

 Entrenamos los modelos que forman el *ensemble*. Mostramos la precisión de cada modelo y el resultado del *Ensemble* comparando el uso de *voting=hard* y *voting=soft*

In [ ]:
voting_clf = VotingClassifier(
    estimators=[('lr', log_clf), ('rf', rnd_clf), ('svc', svm_clf), 
                ('tree1', tree1_clf), ('tree2', tree2_clf)],
    voting='hard')

for clf in (log_clf, rnd_clf, svm_clf, tree1_clf, tree2_clf):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(f'La precisión del método {clf.__class__.__name__} es {accuracy_score(y_test, y_pred)}')

voting_clf.fit(X_train, y_train)
voting_clf.predict(X_test)
print(f'\nLa precisión del "ensemble" usando votación por mayoría es {accuracy_score(y_test, y_pred)}')


voting_clf = VotingClassifier(
    estimators=[('lr', log_clf), ('rf', rnd_clf), ('tree1', tree1_clf), ('tree2', tree2_clf)],
    voting='soft')

voting_clf.fit(X_train, y_train)
voting_clf.predict(X_test)
print(f'La precisión del "ensemble" usando votación ponderada es {accuracy_score(y_test, y_pred)}')

## Regression Gradient Boost

Para ver como se comportan los métodos de **Aprendizaje Conjunto Secuenciales** vamos a implementar un regresor formado por varios **Árboles de Regresión** donde cada uno se entrenará con el error cometido por el anterior. 

Primero lo haremos de forma *manual* añadiendo arbol a arbol a la predicción para ver el efecto de cada uno de ellos y despues usando un modelo de la clase `GradientBoostingRegressor` que crearemos con los parámetros deseados.

In [ ]:
np.random.seed(42)
m = 300
X = np.random.rand(m,1)
y = np.sin(2*np.pi*X)
y = y + np.random.randn(m,1) / 10

plt.plot(X, y, "b.")
plt.xlim(-0.01,1.01)
plt.ylim(-1.4,1.4)
plt.xlabel("X", fontsize=10)
plt.ylabel("y", fontsize=10, rotation= 0)

In [ ]:
tree_reg_1 = DecisionTreeRegressor(max_depth=2)
tree_reg_1.fit(X, y)

y2 = y - tree_reg_1.predict(X)[:,np.newaxis] 
tree_reg_2 = DecisionTreeRegressor(max_depth=2)
tree_reg_2.fit(X, y2)

y3 = y2 - tree_reg_2.predict(X)[:,np.newaxis] # Convierto el vector de predicciones en una matriz
tree_reg_3 = DecisionTreeRegressor(max_depth=2)
tree_reg_3.fit(X, y3)
# 
y4 = y3 - tree_reg_3.predict(X)[:,np.newaxis] # Convierto el vector de predicciones en una matriz
tree_reg_4 = DecisionTreeRegressor(max_depth=2)
tree_reg_4.fit(X, y4)

y5 = y4 - tree_reg_4.predict(X)[:,np.newaxis] # Convierto el vector de predicciones en una matriz
tree_reg_5 = DecisionTreeRegressor(max_depth=2)
tree_reg_5.fit(X, y5)

X_test = np.linspace(0,1,400).reshape(400,1)
y_1 = tree_reg_1.predict(X_test)
y_2 = tree_reg_2.predict(X_test)
y_3 = tree_reg_3.predict(X_test)
y_4 = tree_reg_4.predict(X_test)
y_5 = tree_reg_5.predict(X_test)

print(y_1.shape, y_2.shape)

plt.figure()
plt.plot(X, y, "b.", label="Datos de entrenamiento")
plt.plot(X_test, y_1, "r--", label="Predicción árbol 1", linewidth=1)
plt.plot(X_test, y_1+y_2, "r--", label="Predicción árbol 1 + corrección árbol 2", linewidth=1)
plt.plot(X_test, y_1+y_2+y_3, "r--", label="Predicción árbol 1 + corrección árbol 2 + corrección árbol 3", linewidth=1)
plt.plot(X_test, y_1+y_2+y_3+y_4+y_5, color="green", label="Predicción árbol 1 + corrección árbol 2 + ... + corrección árbol 5", linewidth=2)
plt.legend(loc="upper right", fontsize=8)
plt.ylim(-1.2,2)
plt.figure()

`GradientBoostingRegressor` es un modelo de aprendizaje supervisado en scikit-learn que utiliza *Gradient Boosting* para la regresión. Construye un conjunto de árboles de decisión secuenciales, donde cada nuevo árbol intenta corregir los errores del anterior.

* Se entrena un árbol de decisión inicial para hacer predicciones.
* Se calculan los errores residuales entre las predicciones y los valores reales.
* Se entrena un nuevo árbol para predecir esos errores residuales en lugar del valor original.
* Se repite el proceso hasta completar el número de árboles (n_estimators).
* La predicción final es la suma de todas las correcciones aplicadas en cada iteración.


El parámetro diferente respecto a los modelos entrenados en paralelo es

* `learning_rate` Qué tanto cada nuevo árbol corrige los errores (0.1 por defecto). Un valor más bajo requiere más árboles, pero evita sobreajuste.


Después de entrenar el modelo con fit(X, y), los árboles de decisión individuales se almacenan en el atributo:
```python
gbrt.estimators_
```


In [ ]:

n_estimators_t = 100 
# Entrenamos un Gradient Boosting Regressor con 100 árboles
gbrt = GradientBoostingRegressor(n_estimators=n_estimators_t, max_depth=2, learning_rate=0.1, random_state=42)
gbrt.fit(X, y.ravel())  # Convertimos y a 1D con .ravel() para evitar warnings

# Puntos de prueba para visualizar la evolución
x_test = np.linspace(-0.01, 1.01, 500).reshape(-1, 1)

# Inicializamos las predicciones con el modelo base (promedio de y)
y_pred_cumulative = np.full(x_test.shape, gbrt.init_.predict([[0]]))  # Valor inicial del modelo

plt.figure()

# Iteramos sobre los árboles en estimators_
for i, tree in enumerate(gbrt.estimators_[:, 0], 1):  # [:, 0] porque estimators_ es (n_estimators, 1)
    # Reshape de la predicción para asegurar que tiene la misma forma que y_pred_cumulative
    y_pred_cumulative += gbrt.learning_rate * tree.predict(x_test).reshape(-1, 1)  # Aseguramos que sea un array columna

    # Graficamos la predicción tras este árbol
    if ((i+1)%20==0) and (i != n_estimators_t-1):
        plt.plot(x_test, y_pred_cumulative, "r--", label=f"Árbol {i+1}", linewidth=1)
    elif i == n_estimators_t-1:
        print(i)
        plt.plot(x_test, y_pred_cumulative, "r", label=f"Árbol {i+1}", linewidth=2)
    

# Graficamos los datos originales
plt.plot(X, y, "b.", label="Datos de entrenamiento")

plt.xlim(-0.01, 1.01)
plt.ylim(-1.4, 1.4)
plt.xlabel("X", fontsize=10)
plt.ylabel("y", fontsize=10, rotation=0)
plt.title("Evolución de las predicciones en Gradient Boosting", fontsize=12)
plt.legend()
plt.show()

